# Лаборатория 1.2. Первый разговор с настоящей моделью

**Что мы сделаем:** обратимся к настоящей языковой модели из кода, разберём по косточкам,
что уходит и что приходит, покрутим «ручку случайности» и своими глазами увидим, как
модель уверенно выдумывает несуществующее.

**Что понадобится:** код класса — его даёт учитель. Это не ключ и не пароль:
настоящий ключ лежит на школьном сервере, а код класса просто говорит серверу,
что запрос от нас, а не от случайного человека из интернета.

## Шаг 0. Библиотека для общения с моделью

`openai` — официальная библиотека для обращения к моделям. Название историческое:
по этому же протоколу работают почти все модели, не только OpenAI. Наш школьный
сервер тоже говорит на нём, поэтому библиотека подходит без переделок.

In [ ]:
!pip -q install openai

## Шаг 1. Подключаемся

Две вещи, которые нужны любому обращению к модели:

* **адрес** (`base_url`) — куда стучаться. У нас это школьный сервер, а он уже сам
  передаёт запрос дальше, настоящей модели, и подставляет настоящий ключ;
* **код класса** — чтобы сервер понял, что запрос от учеников.

Код класса нельзя писать прямо в ячейке: ноутбук могут увидеть другие люди.
Поэтому мы либо берём его из «Секретов» Colab (значок ключа на левой панели,
имя секрета `AI9_KOD`), либо спрашиваем при запуске — введённое не будет видно.

In [ ]:
import getpass
import os
from pprint import pprint

from openai import OpenAI

ADRES = "https://ai9.adelfos.ru/api/v1"
MODEL = "qwen/qwen3.7-flash"

try:
    from google.colab import userdata      # если запущено в Colab
    KOD_KLASSA = userdata.get("AI9_KOD")
except Exception:
    KOD_KLASSA = os.environ.get("AI9_KOD") or getpass.getpass("Код класса: ")

client = OpenAI(base_url=ADRES, api_key=KOD_KLASSA)
print("Подключились. Модель:", MODEL)

## Шаг 2. Первый запрос и что в нём происходит

Разберём каждую часть:

* `messages` — история разговора списком. У каждого сообщения есть **роль**:
  * `system` — инструкция «кто ты и как себя веди», её пишет разработчик;
  * `user` — то, что сказал человек;
  * `assistant` — то, что ответила модель (появится в истории потом).
* `temperature` — «ручка случайности», с ней поиграем на шаге 3.
* `max_tokens` — потолок длины ответа. Ставим не «побольше на всякий случай»,
  а столько, сколько нужно: за токены платят.

In [ ]:
soobshcheniya = [
    {"role": "system", "content": "Ты помощник для школьников. Отвечай коротко и просто."},
    {"role": "user", "content": "Объясни в двух предложениях, что такое искусственный интеллект."},
]

print("ЧТО МЫ СПРАШИВАЕМ (уходит на сервер):")
pprint(soobshcheniya, width=100, sort_dicts=False)

otvet = client.chat.completions.create(
    model=MODEL,
    messages=soobshcheniya,
    temperature=0.7,
    max_tokens=200,
)

print("\nЧТО ПРИШЛО ОТ МОДЕЛИ (полный ответ):")
pprint(otvet.model_dump(), width=100, sort_dicts=False)

**Загляни в поле `model` в этом ответе.** Скорее всего, там написано **не то**, что мы
просили в переменной `MODEL`.

Это не ошибка. Между тобой и моделью стоит школьный сервер, и если заказанная модель
сейчас занята или отвечает отказом, он молча берёт следующую из разрешённого списка —
чтобы урок не прервался на середине. Какая модель ответила на самом деле, всегда видно
в поле `model`.

Запомни это как общее правило: **сверяй, что ты получил, а не только что заказал.**
В теме 5 мы увидим, к чему приводит привычка верить, что система работает так, как
задумано, без проверки.

Теперь заглянем внутрь ответа. Кроме текста там есть служебные данные — те самые
токены из прошлой лаборатории, только теперь настоящие, посчитанные сервером.

In [ ]:
print("Роль ответившего:", otvet.choices[0].message.role)
print("Почему модель остановилась:", otvet.choices[0].finish_reason)
print()
print("Токенов ушло (запрос):   ", otvet.usage.prompt_tokens)
print("Токенов пришло (ответ):  ", otvet.usage.completion_tokens)
print("Всего:                   ", otvet.usage.total_tokens)

`finish_reason` стоит запомнить — он объясняет, почему ответ закончился:

* `stop` — модель сама решила, что сказала всё;
* `length` — упёрлись в `max_tokens`, ответ **оборван на полуслове**.

Если увидишь `length`, это не «модель так захотела», это мы поставили слишком
маленький потолок. Попробуй поставить `max_tokens=20` и запустить снова.

## Шаг 3. Ручка случайности

Модель не выбирает следующий кусочек текста жёстко: она смотрит на все варианты
продолжения и выбирает один с учётом их вероятностей. `temperature` управляет тем,
насколько охотно она берёт менее очевидные варианты.

Проверим это честным экспериментом: зададим **один и тот же** вопрос по три раза
при низкой и при высокой температуре и сравним ответы.

In [ ]:
def sprosit(vopros, temperatura, max_tokens=60):
    """Один запрос к модели. Возвращает только текст ответа."""
    otvet = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": vopros}],
        temperature=temperatura,
        max_tokens=max_tokens,
    )
    # Изредка модель возвращает пустой ответ — тогда честно об этом скажем,
    # а не уроним всю лабораторную.
    tekst = otvet.choices[0].message.content or ""
    return tekst.strip() or "(модель вернула пустой ответ — попробуй запустить ячейку ещё раз)"


VOPROS = "Придумай название для школьного кружка робототехники. Ответь только названием."

print(f"Вопрос, который мы задаём шесть раз:\n  {VOPROS}\n")
for temperatura in [0.0, 1.5]:
    print(f"=== температура {temperatura} ===")
    for popytka in range(1, 4):
        print(f"  {popytka}. {sprosit(VOPROS, temperatura)}")
    print()

**Что должно было получиться:** при температуре 0 три ответа почти или полностью
одинаковые, при 1.5 — разные, иногда странные.

Отсюда правило выбора, которым пользуются инженеры:

| Задача | Температура | Почему |
|---|---|---|
| Достать номер телефона из письма | около 0 | правильный ответ ровно один, выдумывать нечего |
| Перевести текст, написать код | 0–0.3 | нужна точность и повторяемость |
| Обычный разговор, объяснение | 0.7 | живой текст без чепухи |
| Придумать идеи, названия, сюжет | 1.0–1.5 | нужно разнообразие |

И важная тонкость: даже при температуре 0 ответ не обязан быть **абсолютно** одинаковым
каждый раз. Внутри складываются миллиарды дробных чисел, и мельчайшие расхождения иногда
меняют выбор. «Температура 0» означает «почти всегда одно и то же», а не «железно одно и то же».

## Шаг 4. Как модель выдумывает

Сейчас будет главный опыт всей темы. Спросим модель о том, чего не существует, —
и посмотрим, признается ли она.

In [ ]:
vydumannaya_kniga = (
    "Перескажи в трёх предложениях сюжет книги «Синие тени Заозёрья» "
    "писателя Аркадия Мельникова, изданной в 1974 году."
)

print("Вопрос, который мы задаём модели:")
pprint(vydumannaya_kniga, width=100)
print()
print("Ответ модели:")
print(sprosit(vydumannaya_kniga, temperatura=0.7, max_tokens=250))

Такой книги и такого писателя не существует — я их придумал, когда писал этот урок.

Скорее всего, ты только что прочитал уверенный пересказ несуществующей книги: с сюжетом,
героями и темами. Это и есть **галлюцинация**.

Теперь — приём, который стоит унести из этого урока. Спросим **то же самое ещё раз**
и сравним два ответа.

In [ ]:
otvet_1 = sprosit(vydumannaya_kniga, temperatura=0.7, max_tokens=200)
otvet_2 = sprosit(vydumannaya_kniga, temperatura=0.7, max_tokens=200)

print("Вопрос дважды:", repr(vydumannaya_kniga), "\n")
print("ПЕРВЫЙ ОТВЕТ:\n", otvet_1, "\n")
print("ВТОРОЙ ОТВЕТ:\n", otvet_2)

Сравни их. Если книга настоящая, оба ответа скажут примерно одно и то же: место действия,
героев, время. А про выдуманную книгу модель каждый раз сочиняет **заново** — и версии
расходятся: то Карелия, то Псковщина, то тридцатые годы, то послевоенные.

Вот тебе бесплатный детектор выдумки, которым можно пользоваться всю жизнь:
**спроси одно и то же дважды и сравни.** Совпало — вероятно, модель это действительно
«знает». Разошлось — она сочиняет.

**Почему так происходит.** Модель не хранит внутри полочку с фактами, которую можно
проверить. Она умеет одно: продолжать текст правдоподобно. Вопрос про несуществующую
книгу для неё ничем не отличается от вопроса про настоящую — в обоих случаях она делает
одно и то же движение.

Это **не ложь**: врать — значит знать правду и говорить другое. Модель не знает.

## Шаг 5. Проверим, спасают ли уговоры

Кажется, что достаточно попросить: «не знаешь — так и скажи». Проверим это честно,
а не на словах. Три попытки, от мягкой к самой прямой.

In [ ]:
popytki = {
    "1. Разрешаем не знать": [
        {"role": "user", "content":
            "Ответь на вопрос. Если не уверен, что такая книга существует, напиши "
            "«я не знаю такой книги» и не придумывай сюжет.\n\n"
            "Вопрос: о чём книга «Синие тени Заозёрья» Аркадия Мельникова, 1974 год?"},
    ],
    "2. Строгая инструкция в system": [
        {"role": "system", "content":
            "Ты честный справочник. Если не уверен в существовании книги, фильма или "
            "человека, отвечай ровно: «Я не знаю такого». Никогда не придумывай факты."},
        {"role": "user", "content": "О чём книга «Синие тени Заозёрья» Аркадия Мельникова, 1974 год?"},
    ],
    "3. Прямой вопрос о существовании": [
        {"role": "system", "content": "Отвечай одним словом: ДА или НЕТ."},
        {"role": "user", "content":
            "Существует ли книга «Синие тени Заозёрья» писателя Аркадия Мельникова 1974 года? "
            "Если не уверен — отвечай НЕТ."},
    ],
}

for nazvanie, messages in popytki.items():
    otvet = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.0, max_tokens=200,
    )
    tekst = (otvet.choices[0].message.content or "").strip()
    print(f"--- {nazvanie} ---")
    print("Что мы отправили:")
    pprint(messages, width=100, sort_dicts=False)
    print("Что нам ответило:", tekst[:300], "\n")

**Скорее всего, не сработало ни разу.** Модель снова сочинила сюжет, а на прямой вопрос
«существует ли такая книга» ответила «ДА».

Это главный вывод лаборатории, и он важнее всех предыдущих:

> Просьба к модели — не защита. Модель не выполняет инструкции, она продолжает текст
> наиболее правдоподобным образом. Иногда просьба помогает, но полагаться на неё нельзя.

Эта мысль вернётся ещё дважды: в теме 4 (агенту границы ставят кодом, а не просьбой)
и в теме 6 (от обмана защищает устройство системы, а не вежливая инструкция).

Что же тогда работает по-настоящему:

1. **Дать модели нужные документы прямо в запрос** — тогда ей не нужно ничего вспоминать.
   Это тема 2, и там ты увидишь, как тот же вопрос получает честное «в документах этого нет».
2. **Проверять ответы измеримо**, а не на глаз — тема 5.
3. **Спросить дважды и сравнить** — приём из шага 4, доступен прямо сейчас.

## Попробуй сам

1. В шаге 2 поставь `max_tokens=20`. Что стало с `finish_reason`?
2. Убери сообщение с ролью `system` и задай тот же вопрос. Ответ изменился по стилю?
3. В шаге 3 замени вопрос на «Столица Франции?» и снова сравни температуры 0 и 1.5.
   Почему здесь разброс намного меньше, чем с названием кружка?
4. Придумай свою несуществующую вещь (фильм, игру, человека) и проверь, выдумает ли
   модель. Спроси дважды — разойдутся ли ответы?
5. Возьми настоящую известную книгу и спроси о ней дважды. Насколько совпали ответы?
   Так ты увидишь разницу между «знает» и «сочиняет» своими глазами.

## Что унести с собой

* Запрос к модели — это список сообщений с ролями: `system`, `user`, `assistant`.
* `finish_reason` показывает, закончила модель сама или её обрезали по `max_tokens`.
* **Температура** управляет разбросом: около 0 — предсказуемо, 1.5 — разнообразно.
* **Галлюцинация** — не ложь и не поломка, а прямое следствие того, что модель
  продолжает текст правдоподобно даже там, где фактов не знает.
* Уговоры («не знаешь — так и скажи») ненадёжны: мы проверили три варианта, и модель
  всё равно сочиняла, а на прямой вопрос о существовании книги ответила «ДА».
  **Просьба к модели — не защита.**
* Зато работает простой приём: **спроси дважды и сравни**. Разошлось — сочиняет.